In [2]:
%pip install duckdb
import duckdb

Note: you may need to restart the kernel to use updated packages.


In [31]:
result = duckdb.sql("""
    SELECT *
    FROM coffee_shop.csv
    ORDER BY transaction_id
    LIMIT 10;
""")

result

┌────────────────┬──────────────────┬──────────────────┬──────────┬─────────────────┬────────────┬─────────────────┬────────────┬────────────┬────────────────────┬───────────────────────┬──────────────────────────┬─────────────┬────────────┬──────────┬───────┬───────┬─────────────┐
│ transaction_id │ transaction_date │ transaction_time │ store_id │ store_location  │ product_id │ transaction_qty │ unit_price │ Total_Bill │  product_category  │     product_type      │      product_detail      │    Size     │ Month Name │ Day Name │ Hour  │ Month │ Day of Week │
│     int64      │       date       │       time       │  int64   │     varchar     │   int64    │      int64      │   double   │   double   │      varchar       │        varchar        │         varchar          │   varchar   │  varchar   │ varchar  │ int64 │ int64 │    int64    │
├────────────────┼──────────────────┼──────────────────┼──────────┼─────────────────┼────────────┼─────────────────┼────────────┼────────────┼─────────

### Question 1: What is the revenue generated by each product category?

In [7]:
result = duckdb.sql("""
    SELECT product_category, 
        SUM(transaction_qty) AS qty,
        ROUND(SUM(total_bill), 2) AS revenue
    FROM coffee_shop.csv
    GROUP BY product_category
    ORDER BY revenue DESC;
""")

result

┌────────────────────┬────────┬───────────┐
│  product_category  │  qty   │  revenue  │
│      varchar       │ int128 │  double   │
├────────────────────┼────────┼───────────┤
│ Coffee             │  89250 │ 269952.45 │
│ Tea                │  69737 │ 196405.95 │
│ Bakery             │  23214 │  82315.64 │
│ Drinking Chocolate │  17457 │   72416.0 │
│ Coffee beans       │   1828 │  40085.25 │
│ Branded            │    776 │   13607.0 │
│ Loose Tea          │   1210 │   11213.6 │
│ Flavours           │  10511 │    8408.8 │
│ Packaged Chocolate │    487 │   4407.64 │
└────────────────────┴────────┴───────────┘

Total_Bill = transactaion_qty * unit_price

### Question 2: What are the 3 most frequently ordered product types in each category?

In [10]:
result = duckdb.sql("""
    WITH temp AS (SELECT product_category, product_type,
        SUM(transaction_qty) AS qty,
        DENSE_RANK() OVER(PARTITION BY product_category ORDER BY qty DESC) AS rank
    FROM coffee_shop.csv
    GROUP BY product_category, product_type
    ORDER BY product_category)
    
    SELECT product_category, product_type, qty, rank
    FROM temp
    WHERE rank <= 3;
""")

result

┌────────────────────┬───────────────────────┬────────┬───────┐
│  product_category  │     product_type      │  qty   │ rank  │
│      varchar       │        varchar        │ int128 │ int64 │
├────────────────────┼───────────────────────┼────────┼───────┤
│ Bakery             │ Scone                 │  10465 │     1 │
│ Bakery             │ Pastry                │   6961 │     2 │
│ Bakery             │ Biscotti              │   5788 │     3 │
│ Branded            │ Housewares            │    555 │     1 │
│ Branded            │ Clothing              │    221 │     2 │
│ Coffee             │ Gourmet brewed coffee │  25973 │     1 │
│ Coffee             │ Barista Espresso      │  24943 │     2 │
│ Coffee             │ Organic brewed coffee │  13012 │     3 │
│ Coffee beans       │ Organic Beans         │    420 │     1 │
│ Coffee beans       │ Premium Beans         │    406 │     2 │
│ Coffee beans       │ Gourmet Beans         │    366 │     3 │
│ Drinking Chocolate │ Hot chocolate    

### Question 3: What is the average bill amount?

In [21]:
result = duckdb.sql("""
    SELECT ROUND(AVG(total_bill), 2) AS avg_bill, MEDIAN(total_bill),
        MAX(total_bill) AS highest_bill, MIN(total_bill) AS lowest_bill
    FROM coffee_shop.csv
""")

result

┌──────────┬────────────────────┬──────────────┬─────────────┐
│ avg_bill │ median(total_bill) │ highest_bill │ lowest_bill │
│  double  │       double       │    double    │   double    │
├──────────┼────────────────────┼──────────────┼─────────────┤
│     4.69 │               3.75 │        360.0 │         0.8 │
└──────────┴────────────────────┴──────────────┴─────────────┘

### Question 4: What was the revenue generated in each month in 2023?

In [29]:
result = duckdb.sql("""
    SELECT month, 
        ROUND(SUM(total_bill), 2) AS revenue
    FROM coffee_shop.csv
    WHERE EXTRACT("year" FROM transaction_date) = 2023
    GROUP BY month
    ORDER BY month;
""")

result

┌───────┬───────────┐
│ Month │  revenue  │
│ int64 │  double   │
├───────┼───────────┤
│     1 │  81677.74 │
│     2 │  76145.19 │
│     3 │  98834.68 │
│     4 │ 118941.08 │
│     5 │ 156727.76 │
│     6 │ 166485.88 │
└───────┴───────────┘

### Question 5: How do the results from each month compare to those from the previous month?

In [94]:
result = duckdb.sql("""
    WITH temp1 AS (
        SELECT month,
        ROUND(SUM(total_bill), 2) AS revenue 
        FROM coffee_shop.csv
        GROUP BY month
    ),
    
    temp2 AS (
        SELECT month, revenue,
            LAG(revenue, 1, 0) OVER(ORDER BY month) AS previous_month_revenue
        FROM temp1
    )
    
    SELECT month, revenue, previous_month_revenue,
        ROUND((revenue - previous_month_revenue) / previous_month_revenue, 3) AS change
    FROM temp2;
""")

result

┌───────┬───────────┬────────────────────────┬────────┐
│ Month │  revenue  │ previous_month_revenue │ change │
│ int64 │  double   │         double         │ double │
├───────┼───────────┼────────────────────────┼────────┤
│     1 │  81677.74 │                    0.0 │    inf │
│     2 │  76145.19 │               81677.74 │ -0.068 │
│     3 │  98834.68 │               76145.19 │  0.298 │
│     4 │ 118941.08 │               98834.68 │  0.203 │
│     5 │ 156727.76 │              118941.08 │  0.318 │
│     6 │ 166485.88 │              156727.76 │  0.062 │
└───────┴───────────┴────────────────────────┴────────┘